# RunPod JupyterLab 기반 Qwen2.5 한국어 SFT 전체 실습

이 Notebook은 다음 전체 흐름을 한 파일에서 실행합니다.

1. RunPod GPU 및 CUDA 확인
2. 필수 라이브러리 설치
3. 프로젝트 경로와 환경변수 설정
4. 한국어 상담 데이터 생성
5. 데이터 검증 및 학습/검증 분할
6. Tokenizer 로드
7. Qwen2.5 모델 4bit 양자화 로드
8. LoRA Adapter 설정
9. TRL `SFTTrainer` 구성
10. Supervised Fine-Tuning 실행
11. 평가 결과와 Adapter 저장
12. 학습 직후 추론
13. 저장된 Adapter 재로드
14. 대화형 질문 테스트

> 셀은 위에서 아래로 순서대로 실행합니다.  
> RunPod Pod를 중지하기 전에 `/workspace` 아래 결과를 외부 저장소로 백업해야 합니다.

## 1. GPU와 실행 환경 확인

RunPod에서 GPU가 정상 할당되었는지 먼저 확인합니다.

In [ ]:
# 운영체제 명령을 Notebook 셀에서 실행하여 NVIDIA GPU 정보를 확인합니다.
!nvidia-smi

# Python 실행 버전을 확인합니다.
!python --version

# 현재 작업 디렉터리를 확인합니다.
!pwd

In [ ]:
# PyTorch를 가져와 CUDA 사용 가능 여부를 확인합니다.
import torch

# 설치된 PyTorch 버전을 출력합니다.
print("PyTorch 버전:", torch.__version__)

# PyTorch가 CUDA GPU를 인식하는지 출력합니다.
print("CUDA 사용 가능:", torch.cuda.is_available())

# CUDA GPU가 정상적으로 인식된 경우 상세 정보를 출력합니다.
if torch.cuda.is_available():
    # 현재 사용할 GPU의 이름을 출력합니다.
    print("GPU 이름:", torch.cuda.get_device_name(0))

    # PyTorch가 사용하는 CUDA 버전을 출력합니다.
    print("PyTorch CUDA 버전:", torch.version.cuda)

    # GPU 전체 메모리를 GB 단위로 변환하여 출력합니다.
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU 전체 메모리: {total_memory_gb:.2f} GB")
else:
    # GPU가 인식되지 않으면 학습을 진행하지 말고 RunPod GPU 설정을 확인합니다.
    raise RuntimeError("CUDA GPU가 인식되지 않습니다. RunPod Pod의 GPU 설정을 확인하세요.")

## 2. 필수 라이브러리 설치

RunPod 템플릿에 이미 설치된 패키지가 있더라도 버전을 맞추기 위해 업데이트합니다.  
설치 후 Kernel 재시작이 필요할 수 있습니다.

In [ ]:
# pip, setuptools, wheel을 먼저 최신 버전으로 업데이트합니다.
!python -m pip install --upgrade pip setuptools wheel

# SFT, QLoRA 및 Hugging Face 모델 사용에 필요한 패키지를 설치합니다.
!python -m pip install --upgrade \
    transformers \
    datasets \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    sentencepiece \
    huggingface-hub \
    tensorboard \
    python-dotenv

In [ ]:
# 설치된 주요 패키지 버전을 확인합니다.
import transformers
import datasets
import trl
import peft
import accelerate
import bitsandbytes

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

## 3. 프로젝트 경로와 환경변수 설정

RunPod의 영구 작업 공간으로 일반적으로 `/workspace`를 사용합니다.

In [ ]:
# 운영체제 환경변수 처리를 위해 os 모듈을 가져옵니다.
import os

# 파일과 디렉터리 경로를 안전하게 관리하기 위해 Path를 가져옵니다.
from pathlib import Path

# Notebook에서 사용할 프로젝트 루트 경로를 지정합니다.
PROJECT_DIR = Path("/workspace/qwen_sft_runpod_jupyterlab")

# 학습 데이터 저장 폴더를 지정합니다.
DATA_DIR = PROJECT_DIR / "data"

# 체크포인트 저장 폴더를 지정합니다.
CHECKPOINT_DIR = PROJECT_DIR / "outputs" / "checkpoints"

# 최종 LoRA Adapter 저장 폴더를 지정합니다.
OUTPUT_DIR = PROJECT_DIR / "outputs" / "qwen2.5-korean-sft-lora"

# 필요한 모든 폴더를 생성합니다.
for directory in [PROJECT_DIR, DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# 기본 모델 이름을 지정합니다.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Hugging Face 토큰이 필요한 모델을 사용할 경우 아래 환경변수에 입력합니다.
# 공개 모델만 사용하는 현재 실습에서는 빈 문자열이어도 됩니다.
HF_TOKEN = os.getenv("HF_TOKEN", "").strip() or None

# 생성된 주요 경로를 출력합니다.
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("HF_TOKEN 설정 여부:", HF_TOKEN is not None)

## 4. 한국어 상담 SFT 데이터 생성

각 샘플은 `system → user → assistant` 순서의 `messages` 형식으로 구성합니다.

In [ ]:
# JSONL 파일 저장을 위해 json 모듈을 가져옵니다.
import json

# 학습/검증 데이터 분할 전 순서를 섞기 위해 random 모듈을 가져옵니다.
import random

# 모든 샘플에 공통으로 적용할 시스템 프롬프트를 정의합니다.
SYSTEM_PROMPT = """
당신은 온라인 쇼핑몰의 고객 상담 AI입니다.
고객의 질문을 정확하게 이해하고 친절하고 간결하게 답변합니다.
확인되지 않은 정책이나 정보를 임의로 만들지 않습니다.
추가 정보가 필요한 경우 고객에게 필요한 정보를 요청합니다.
답변은 원칙적으로 한국어로 작성합니다.
""".strip()

# 사용자 질문과 상담 정답을 튜플 목록으로 정의합니다.
RAW_SAMPLES = [
    ("주문한 상품을 취소하고 싶습니다.", "상품이 아직 배송 준비 전이라면 주문 내역에서 직접 취소할 수 있습니다. 이미 배송이 시작된 경우에는 상품 수령 후 반품을 신청해야 합니다. 정확한 상태를 확인하려면 주문번호를 알려주세요."),
    ("배송은 보통 얼마나 걸리나요?", "일반 배송은 결제 완료 후 평균 2~3영업일 정도 소요됩니다. 도서산간 지역이나 주문량이 많은 기간에는 배송이 지연될 수 있습니다."),
    ("상품을 받았는데 파손되어 있습니다.", "불편을 드려 죄송합니다. 파손된 상품과 포장 상태를 확인할 수 있는 사진을 준비한 뒤 주문번호와 함께 교환 또는 반품을 신청해 주세요. 상품 확인 후 교환이나 환불 절차를 안내해 드리겠습니다."),
    ("반품 배송비는 누가 부담하나요?", "단순 변심에 의한 반품은 일반적으로 고객이 반품 배송비를 부담합니다. 상품 불량, 오배송 또는 파손과 같이 판매자 책임이 있는 경우에는 판매자가 반품 배송비를 부담합니다."),
    ("환불은 언제 처리되나요?", "반품 상품이 판매자에게 도착하고 검수가 완료된 후 환불이 진행됩니다. 카드 결제는 카드사에 따라 승인 취소 반영까지 추가로 3~7영업일이 걸릴 수 있습니다."),
    ("주문번호를 모르는데 주문 조회가 가능한가요?", "회원 주문이라면 로그인 후 주문 내역에서 확인할 수 있습니다. 비회원 주문이라면 주문자 이름, 연락처 또는 결제 정보를 이용해 조회할 수 있습니다."),
    ("결제는 완료됐는데 주문 상태가 결제 대기로 나옵니다.", "결제 정보가 주문 시스템에 반영되는 데 일시적으로 시간이 걸릴 수 있습니다. 잠시 후 주문 내역을 다시 확인해 주세요. 계속 결제 대기로 표시된다면 주문번호와 결제 시간을 알려주세요."),
    ("배송지를 잘못 입력했습니다.", "상품이 배송 준비 전이라면 주문 내역 또는 고객센터를 통해 배송지 변경을 요청할 수 있습니다. 이미 출고된 경우에는 배송지 변경이 제한될 수 있으므로 빠르게 문의해 주세요."),
    ("교환하고 싶은데 어떻게 해야 하나요?", "주문 내역에서 교환할 상품을 선택한 후 교환 사유와 원하는 옵션을 입력해 신청해 주세요. 상품 상태 확인 후 회수 및 재배송 절차가 진행됩니다."),
    ("회원 탈퇴 후 주문 내역도 삭제되나요?", "회원 탈퇴 후에도 전자상거래 관련 법령에 따라 거래 기록 일부는 일정 기간 보관될 수 있습니다. 보관 기간이 종료되면 관련 정책에 따라 안전하게 삭제됩니다."),
    ("쿠폰을 사용하지 못하고 결제했습니다.", "결제가 완료된 주문에는 일반적으로 쿠폰을 추가 적용할 수 없습니다. 상품이 아직 배송 준비 전이라면 기존 주문을 취소한 뒤 쿠폰을 적용하여 다시 주문할 수 있습니다."),
    ("품절된 상품은 언제 다시 입고되나요?", "재입고 일정은 상품과 공급 상황에 따라 달라집니다. 상품 페이지에 재입고 알림 기능이 있다면 신청해 주세요. 정확한 확인을 위해 상품명을 알려주시면 확인 방법을 안내해 드리겠습니다."),
    ("다른 상품이 배송되었습니다.", "오배송으로 불편을 드려 죄송합니다. 받으신 상품의 사진과 주문번호를 준비하여 교환 또는 반품을 신청해 주세요. 확인 후 올바른 상품을 받을 수 있도록 안내해 드리겠습니다."),
    ("현금영수증을 발급받고 싶습니다.", "현금 결제 과정에서 현금영수증 발급 정보를 입력할 수 있습니다. 결제 후 발급이 필요한 경우 주문번호와 결제 정보를 확인한 뒤 고객센터에 요청해 주세요."),
    ("고객센터 운영 시간이 궁금합니다.", "고객센터 운영 시간은 쇼핑몰 정책에 따라 다를 수 있습니다. 사이트의 고객센터 또는 이용 안내 페이지에서 정확한 운영 시간을 확인해 주세요."),
    ("상품 사용 후에도 반품할 수 있나요?", "상품을 사용하여 가치가 감소했거나 재판매가 어려운 상태라면 반품이 제한될 수 있습니다. 다만 상품 자체의 하자가 확인되는 경우에는 별도 절차로 처리될 수 있습니다."),
    ("배송 완료라고 나오는데 상품을 받지 못했습니다.", "먼저 가족, 경비실, 무인 택배함 또는 문 앞에 상품이 보관되었는지 확인해 주세요. 상품이 확인되지 않으면 배송 조회에 표시된 택배사와 기사에게 문의한 뒤 주문번호를 고객센터에 알려주세요."),
    ("결제 수단을 변경하고 싶습니다.", "결제가 완료된 주문의 결제 수단은 직접 변경하기 어렵습니다. 배송 준비 전이라면 주문을 취소한 후 원하는 결제 수단으로 다시 주문해 주세요."),
    ("부분 환불도 가능한가요?", "여러 상품을 함께 주문했다면 일부 상품만 선택하여 반품 및 환불을 신청할 수 있습니다. 다만 할인이나 쿠폰 조건이 변경되면 최종 환불 금액이 달라질 수 있습니다."),
    ("문의 글을 남겼는데 답변이 없습니다.", "문의량이 많으면 답변이 지연될 수 있습니다. 문의 내역에서 접수 상태를 확인해 주세요. 오랫동안 답변이 없다면 문의 날짜와 제목을 알려주시면 확인 방법을 안내해 드리겠습니다."),
]

# 원본 튜플을 Hugging Face 대화형 messages 형식으로 변환합니다.
all_samples = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ]
    }
    for user_text, assistant_text in RAW_SAMPLES
]

# 생성된 전체 샘플 수를 출력합니다.
print("전체 샘플 수:", len(all_samples))

# 첫 번째 샘플을 보기 쉬운 JSON 형식으로 출력합니다.
print(json.dumps(all_samples[0], ensure_ascii=False, indent=2))

In [ ]:
# 한 개의 SFT 샘플 구조를 검사하는 함수를 정의합니다.
def validate_sample(sample: dict) -> None:
    # 최상위 객체에 messages 키가 있는지 확인합니다.
    if "messages" not in sample:
        raise ValueError("messages 키가 없습니다.")

    # messages가 리스트인지 확인합니다.
    if not isinstance(sample["messages"], list):
        raise ValueError("messages 값은 리스트여야 합니다.")

    # system, user, assistant 메시지가 모두 존재하는지 확인합니다.
    if len(sample["messages"]) != 3:
        raise ValueError("각 샘플은 메시지 3개를 포함해야 합니다.")

    # 요구되는 역할 순서를 정의합니다.
    expected_roles = ["system", "user", "assistant"]

    # 실제 메시지를 순회하며 역할과 내용을 검사합니다.
    for message, expected_role in zip(sample["messages"], expected_roles):
        # 현재 메시지가 딕셔너리인지 확인합니다.
        if not isinstance(message, dict):
            raise ValueError("각 메시지는 딕셔너리여야 합니다.")

        # 역할 순서가 올바른지 확인합니다.
        if message.get("role") != expected_role:
            raise ValueError(
                f"역할 순서 오류: 예상={expected_role}, 실제={message.get('role')}"
            )

        # content가 비어 있지 않은 문자열인지 확인합니다.
        content = message.get("content")
        if not isinstance(content, str) or not content.strip():
            raise ValueError(f"{expected_role} content가 비어 있습니다.")


# 모든 샘플을 검증합니다.
for sample in all_samples:
    validate_sample(sample)

# 모든 검증을 통과했다는 메시지를 출력합니다.
print("모든 데이터 구조 검증 완료")

In [ ]:
# 반복 실행해도 동일한 학습/검증 분할을 얻기 위한 난수 생성기를 만듭니다.
random_generator = random.Random(42)

# 원본 목록을 변경하지 않도록 복사본을 만듭니다.
shuffled_samples = all_samples.copy()

# 복사한 데이터의 순서를 무작위로 섞습니다.
random_generator.shuffle(shuffled_samples)

# 전체 데이터의 80% 지점을 분할 인덱스로 계산합니다.
split_index = int(len(shuffled_samples) * 0.8)

# 앞쪽 80%를 학습 데이터로 선택합니다.
train_samples = shuffled_samples[:split_index]

# 나머지 20%를 검증 데이터로 선택합니다.
valid_samples = shuffled_samples[split_index:]

# 학습 및 검증 파일의 경로를 지정합니다.
TRAIN_FILE = DATA_DIR / "train.jsonl"
VALID_FILE = DATA_DIR / "valid.jsonl"


# JSONL 형식으로 데이터를 저장하는 함수를 정의합니다.
def save_jsonl(samples: list[dict], file_path: Path) -> None:
    # UTF-8 쓰기 모드로 파일을 엽니다.
    with file_path.open("w", encoding="utf-8") as file:
        # 각 샘플을 한 줄씩 저장합니다.
        for sample in samples:
            # ensure_ascii=False를 적용해 한글을 그대로 기록합니다.
            file.write(json.dumps(sample, ensure_ascii=False) + "\n")


# 학습 데이터를 저장합니다.
save_jsonl(train_samples, TRAIN_FILE)

# 검증 데이터를 저장합니다.
save_jsonl(valid_samples, VALID_FILE)

# 저장 결과를 출력합니다.
print("학습 데이터 수:", len(train_samples))
print("검증 데이터 수:", len(valid_samples))
print("학습 파일:", TRAIN_FILE)
print("검증 파일:", VALID_FILE)

## 5. Hugging Face Dataset으로 로드

In [ ]:
# JSONL 파일을 Hugging Face Dataset으로 읽기 위해 load_dataset을 가져옵니다.
from datasets import load_dataset

# 학습/검증 파일 경로를 분할 이름과 연결합니다.
data_files = {
    "train": str(TRAIN_FILE),
    "validation": str(VALID_FILE),
}

# JSON 데이터셋을 DatasetDict 형식으로 로드합니다.
dataset_dict = load_dataset(
    "json",
    data_files=data_files,
)

# 학습 분할을 가져옵니다.
train_dataset = dataset_dict["train"]

# 검증 분할을 가져옵니다.
valid_dataset = dataset_dict["validation"]

# 데이터셋 정보를 출력합니다.
print(train_dataset)
print(valid_dataset)
print("컬럼:", train_dataset.column_names)

# 첫 번째 학습 샘플을 출력합니다.
print(json.dumps(train_dataset[0], ensure_ascii=False, indent=2))

## 6. Tokenizer와 4bit 양자화 모델 로드

In [ ]:
# Hugging Face 모델, Tokenizer 및 양자화 설정 클래스를 가져옵니다.
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)

# 동일한 난수 흐름을 사용하도록 시드를 고정합니다.
SEED = 42
set_seed(SEED)

# GPU가 BF16을 지원하면 BF16을 사용하고, 그렇지 않으면 FP16을 사용합니다.
COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

# 선택된 연산 자료형을 출력합니다.
print("선택된 연산 타입:", COMPUTE_DTYPE)

In [ ]:
# 기본 모델에 맞는 Tokenizer를 Hugging Face Hub에서 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN,
)

# Padding Token이 없는 경우 EOS Token을 대신 사용합니다.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Causal Language Model 학습에 적합하도록 오른쪽 Padding을 사용합니다.
tokenizer.padding_side = "right"

# Tokenizer의 주요 특수 토큰 정보를 출력합니다.
print("PAD Token:", tokenizer.pad_token)
print("PAD Token ID:", tokenizer.pad_token_id)
print("EOS Token:", tokenizer.eos_token)
print("EOS Token ID:", tokenizer.eos_token_id)

In [ ]:
# 4bit QLoRA 학습에 사용할 BitsAndBytes 양자화 설정을 정의합니다.
quantization_config = BitsAndBytesConfig(
    # 모델 가중치를 4bit로 불러와 GPU 메모리 사용량을 줄입니다.
    load_in_4bit=True,

    # QLoRA에서 일반적으로 사용하는 NF4 양자화 방식을 사용합니다.
    bnb_4bit_quant_type="nf4",

    # 양자화 상수를 다시 양자화하여 메모리를 추가로 절약합니다.
    bnb_4bit_use_double_quant=True,

    # 실제 행렬 연산은 BF16 또는 FP16으로 수행합니다.
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

# 생성한 양자화 설정을 출력합니다.
print(quantization_config)

In [ ]:
# Qwen2.5 Causal Language Model을 4bit 양자화 상태로 불러옵니다.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    trust_remote_code=True,
    token=HF_TOKEN,
)

# 모델의 Padding Token ID를 Tokenizer 설정과 맞춥니다.
model.config.pad_token_id = tokenizer.pad_token_id

# Gradient Checkpointing과 충돌할 수 있으므로 학습 중 KV Cache를 끕니다.
model.config.use_cache = False

# 모델이 배치된 장치를 출력합니다.
print("모델 장치:", model.device)

## 7. k-bit 학습 준비와 LoRA 설정

In [ ]:
# PEFT의 LoRA 설정과 k-bit 학습 준비 함수를 가져옵니다.
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

# 양자화 모델이 LoRA 학습에 적합하도록 준비합니다.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# 중간 활성값 저장을 줄이기 위해 Gradient Checkpointing을 활성화합니다.
model.gradient_checkpointing_enable()

# 모델이 정상 준비되었음을 출력합니다.
print("k-bit 학습 준비 완료")

In [ ]:
# Qwen 모델의 Attention 및 MLP Projection Layer에 적용할 LoRA 설정을 정의합니다.
lora_config = LoraConfig(
    # 저차원 행렬의 Rank입니다.
    r=16,

    # LoRA 출력에 적용하는 Scaling 계수입니다.
    lora_alpha=32,

    # LoRA 경로의 과적합을 줄이기 위한 Dropout 비율입니다.
    lora_dropout=0.05,

    # 기존 Linear Layer의 Bias는 학습하지 않습니다.
    bias="none",

    # 다음 토큰 예측형 Causal LM 작업임을 지정합니다.
    task_type="CAUSAL_LM",

    # LoRA Adapter를 적용할 내부 Linear Layer 이름입니다.
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# LoRA 설정을 출력합니다.
print(lora_config)

## 8. SFTTrainer 학습 설정과 생성

In [ ]:
# TRL의 SFT 전용 설정과 Trainer를 가져옵니다.
from trl import SFTConfig, SFTTrainer

# 학습 하이퍼파라미터를 정의합니다.
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

# BF16 사용 여부를 계산합니다.
USE_BF16 = COMPUTE_DTYPE == torch.bfloat16

# FP16 사용 여부를 계산합니다.
USE_FP16 = COMPUTE_DTYPE == torch.float16

# SFTTrainer가 사용할 학습 설정을 생성합니다.
training_config = SFTConfig(
    # 체크포인트와 Trainer 상태를 저장할 경로입니다.
    output_dir=str(CHECKPOINT_DIR),

    # 전체 학습 데이터 반복 횟수입니다.
    num_train_epochs=NUM_TRAIN_EPOCHS,

    # GPU 한 장당 학습 배치 크기입니다.
    per_device_train_batch_size=TRAIN_BATCH_SIZE,

    # GPU 한 장당 검증 배치 크기입니다.
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    # 여러 Step의 Gradient를 누적한 뒤 한 번 갱신합니다.
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # LoRA 파라미터의 학습률입니다.
    learning_rate=LEARNING_RATE,

    # 가중치 감쇠 규제 계수입니다.
    weight_decay=WEIGHT_DECAY,

    # 학습률 Warm-up 비율입니다.
    warmup_ratio=WARMUP_RATIO,

    # Cosine 방식으로 학습률을 감소시킵니다.
    lr_scheduler_type="cosine",

    # 메모리 효율적인 8bit Paged AdamW Optimizer를 사용합니다.
    optim="paged_adamw_8bit",

    # 매 Step마다 학습 로그를 출력합니다.
    logging_steps=1,

    # 첫 Step부터 로그를 기록합니다.
    logging_first_step=True,

    # 각 Epoch 종료 시 검증합니다.
    eval_strategy="epoch",

    # 각 Epoch 종료 시 체크포인트를 저장합니다.
    save_strategy="epoch",

    # 최대 두 개의 체크포인트만 유지합니다.
    save_total_limit=2,

    # 학습 종료 후 eval_loss가 가장 낮은 모델을 복원합니다.
    load_best_model_at_end=True,

    # 최적 모델 판단 기준을 eval_loss로 지정합니다.
    metric_for_best_model="eval_loss",

    # eval_loss는 낮을수록 좋습니다.
    greater_is_better=False,

    # 지원 GPU에서는 BF16 혼합 정밀도를 사용합니다.
    bf16=USE_BF16,

    # BF16 미지원 GPU에서는 FP16 혼합 정밀도를 사용합니다.
    fp16=USE_FP16,

    # GPU 메모리를 절약하기 위해 Gradient Checkpointing을 사용합니다.
    gradient_checkpointing=True,

    # 비재진입 Gradient Checkpointing 방식을 사용합니다.
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # 학습 입력의 최대 Token 길이입니다.
    max_length=MAX_LENGTH,

    # 여러 샘플을 한 시퀀스로 합치는 Packing은 비활성화합니다.
    packing=False,

    # TensorBoard 형식으로 로그를 저장합니다.
    report_to=["tensorboard"],

    # 데이터 전처리에 사용할 프로세스 수입니다.
    dataset_num_proc=1,

    # messages 컬럼이 자동 제거되지 않도록 합니다.
    remove_unused_columns=False,

    # 학습 및 데이터 난수 시드를 고정합니다.
    seed=SEED,
    data_seed=SEED,
)

# 설정을 출력합니다.
print(training_config)

In [ ]:
# 모델, 데이터셋, Tokenizer, LoRA 설정을 연결하여 SFTTrainer를 생성합니다.
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# PEFT 모델에서 실제 학습되는 파라미터 수와 비율을 출력합니다.
trainer.model.print_trainable_parameters()

## 9. Supervised Fine-Tuning 실행

In [ ]:
# 실제 SFT 학습을 시작합니다.
train_result = trainer.train()

# 최종 Training Loss를 출력합니다.
print("최종 Training Loss:", train_result.training_loss)

# 학습 결과 지표를 출력합니다.
print(train_result.metrics)

## 10. 평가 및 결과 저장

In [ ]:
# 검증 데이터셋으로 최종 평가를 실행합니다.
eval_metrics = trainer.evaluate()

# 평가 지표를 출력합니다.
print("평가 결과:")
print(eval_metrics)

In [ ]:
# 가장 좋은 LoRA Adapter를 최종 출력 폴더에 저장합니다.
trainer.save_model(str(OUTPUT_DIR))

# 학습 때 사용한 Tokenizer도 같은 경로에 저장합니다.
tokenizer.save_pretrained(str(OUTPUT_DIR))

# Trainer의 학습 지표를 저장합니다.
trainer.save_metrics("train", train_result.metrics)

# Trainer의 평가 지표를 저장합니다.
trainer.save_metrics("eval", eval_metrics)

# Optimizer, Scheduler 및 Global Step 등의 Trainer 상태를 저장합니다.
trainer.save_state()

# 저장된 파일 목록을 출력합니다.
print("최종 저장 경로:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)

In [ ]:
# 주요 학습 설정과 결과를 별도 JSON 파일로 저장합니다.
training_summary = {
    "base_model": MODEL_NAME,
    "output_dir": str(OUTPUT_DIR),
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "compute_dtype": str(COMPUTE_DTYPE),
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
}

# 요약 파일 경로를 지정합니다.
summary_file = OUTPUT_DIR / "training_summary.json"

# UTF-8 JSON 형식으로 저장합니다.
with summary_file.open("w", encoding="utf-8") as file:
    json.dump(
        training_summary,
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

# 저장 위치를 출력합니다.
print("학습 요약 저장:", summary_file)

## 11. 학습 직후 추론

In [ ]:
# 학습 직후 현재 메모리에 있는 모델로 답변을 생성하는 함수를 정의합니다.
def generate_answer(model, tokenizer, question: str) -> str:
    # 추론 시 KV Cache를 활성화합니다.
    model.config.use_cache = True

    # Dropout 등을 끄기 위해 평가 모드로 전환합니다.
    model.eval()

    # system/user 메시지를 구성합니다.
    messages = [
        {
            "role": "system",
            "content": (
                "당신은 온라인 쇼핑몰의 고객 상담 AI입니다. "
                "고객의 질문에 친절하고 정확하게 답변하세요. "
                "확인되지 않은 사실은 임의로 만들지 마세요."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    # Qwen 전용 Chat Template을 적용해 문자열 프롬프트를 생성합니다.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 문자열 프롬프트를 PyTorch Tensor로 변환합니다.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # 입력 Tensor를 모델이 있는 GPU로 이동합니다.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Gradient 계산 없이 답변 Token을 생성합니다.
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 입력 프롬프트 Token을 제외한 신규 Token만 선택합니다.
    generated_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]

    # 신규 Token을 문자열로 변환합니다.
    answer = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )[0]

    # 앞뒤 공백을 제거한 답변을 반환합니다.
    return answer.strip()

In [ ]:
# 학습 결과를 확인할 테스트 질문을 지정합니다.
test_question = "배송 완료로 나오지만 상품을 받지 못했습니다."

# 학습된 모델로 답변을 생성합니다.
test_answer = generate_answer(
    model=trainer.model,
    tokenizer=tokenizer,
    question=test_question,
)

# 질문과 답변을 출력합니다.
print("[질문]")
print(test_question)
print("\n[답변]")
print(test_answer)

## 12. 메모리 정리 후 저장된 Adapter 재로드

In [ ]:
# 사용하지 않는 객체를 정리하기 위해 gc를 가져옵니다.
import gc

# 기존 Trainer와 모델 참조를 제거합니다.
del trainer
del model

# Python 가비지 컬렉션을 실행합니다.
gc.collect()

# CUDA 캐시 메모리를 해제합니다.
torch.cuda.empty_cache()

# 현재 GPU 메모리 상태를 확인합니다.
!nvidia-smi

In [ ]:
# 저장된 LoRA Adapter를 기본 모델에 연결하기 위해 PeftModel을 가져옵니다.
from peft import PeftModel

# 저장된 Tokenizer를 최종 Adapter 폴더에서 불러옵니다.
reloaded_tokenizer = AutoTokenizer.from_pretrained(
    str(OUTPUT_DIR),
    trust_remote_code=True,
)

# Padding Token이 없으면 EOS Token을 대신 사용합니다.
if reloaded_tokenizer.pad_token is None:
    reloaded_tokenizer.pad_token = reloaded_tokenizer.eos_token

# 기본 모델을 다시 4bit 양자화 상태로 불러옵니다.
reloaded_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    trust_remote_code=True,
    token=HF_TOKEN,
)

# 기본 모델과 Tokenizer의 Padding Token ID를 맞춥니다.
reloaded_base_model.config.pad_token_id = reloaded_tokenizer.pad_token_id

# 기본 모델 위에 저장된 LoRA Adapter를 연결합니다.
reloaded_model = PeftModel.from_pretrained(
    reloaded_base_model,
    str(OUTPUT_DIR),
)

# 추론용 KV Cache를 활성화합니다.
reloaded_model.config.use_cache = True

# 모델을 평가 모드로 전환합니다.
reloaded_model.eval()

# 재로드 완료를 출력합니다.
print("저장된 LoRA Adapter 재로드 완료")

In [ ]:
# 재로드된 모델로 다시 답변을 생성합니다.
reload_question = "결제 수단을 변경하고 싶습니다."

reload_answer = generate_answer(
    model=reloaded_model,
    tokenizer=reloaded_tokenizer,
    question=reload_question,
)

print("[질문]")
print(reload_question)
print("\n[답변]")
print(reload_answer)

## 13. Notebook 대화형 질문 함수

아래 셀에서 `question` 값만 변경해 반복 실행합니다.

In [ ]:
# 테스트할 사용자 질문을 입력합니다.
question = "상품을 받았는데 파손되어 있습니다."

# 저장된 Adapter가 적용된 모델로 답변을 생성합니다.
answer = generate_answer(
    model=reloaded_model,
    tokenizer=reloaded_tokenizer,
    question=question,
)

# 최종 결과를 출력합니다.
print("질문:", question)
print("답변:", answer)

## 14. TensorBoard 실행

별도 Terminal 또는 새 Notebook 셀에서 아래 명령을 실행합니다.

```bash
tensorboard --logdir /workspace/qwen_sft_runpod_jupyterlab/outputs/checkpoints/runs --host 0.0.0.0 --port 6006
```

RunPod의 **Connect → HTTP Services**에서 6006 포트를 열어 확인합니다.

In [ ]:
# 학습 결과 폴더의 전체 파일 구조를 확인합니다.
!find /workspace/qwen_sft_runpod_jupyterlab -maxdepth 4 -type f | sort

## 15. 결과 압축 및 백업

RunPod Pod를 중지하거나 삭제하기 전에 결과를 ZIP으로 압축합니다.

In [ ]:
# 프로젝트 폴더 전체를 ZIP 파일로 압축합니다.
!cd /workspace && zip -r qwen_sft_runpod_jupyterlab_result.zip qwen_sft_runpod_jupyterlab

# 생성된 ZIP 파일 크기와 경로를 확인합니다.
!ls -lh /workspace/qwen_sft_runpod_jupyterlab_result.zip

## 오류 대응

### CUDA Out of Memory

학습 설정 셀에서 다음 값을 줄입니다.

```python
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
```

LoRA Rank도 `r=8`로 낮출 수 있습니다.

### `eval_strategy` 인자 오류

설치된 TRL 또는 Transformers가 오래된 경우입니다.

```bash
pip install -U transformers trl peft accelerate
```

그래도 구버전을 유지해야 한다면 `eval_strategy="epoch"`를  
`evaluation_strategy="epoch"`로 바꿔야 할 수 있습니다.

### Kernel 재시작

패키지를 새로 설치한 직후 Import 오류가 발생하면 JupyterLab 메뉴에서:

```text
Kernel → Restart Kernel
```

을 실행하고 Notebook을 처음부터 다시 실행합니다.